```mermaid
flowchart TD
    A([START]) --> B[generate_report_node]
    B --> C[human_review_node: interrupt]

    C --> D{Human decision}
    D -- reject --> G([END - không lưu])
    D -- approve --> E[save_report_node]
    D -- edit --> F[apply_feedback_node]
    F --> B

    E --> H([END - đã lưu report])
```

In [ ]:
# pip install -U langgraph

from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


class State(TypedDict):
    request: str
    draft: str
    approved: bool
    saved_path: str
    reviewer_note: str
    revision_count: int
    feedback: str
    last_decision: str


def generate_report_node(state: State):
    """
    Sinh lại report dựa trên request + feedback gần nhất.
    Trong thực tế bạn có thể gọi LLM ở đây.
    """
    base_request = state["request"]
    revision_count = state.get("revision_count", 0)
    feedback = state.get("feedback", "").strip()

    if revision_count == 0:
        draft = (
            "BÁO CÁO TUẦN - BẢN NHÁP LẦN 1\n"
            f"- Yêu cầu: {base_request}\n"
            "- Tiến độ: Hoàn thành 80%\n"
            "- Rủi ro: Cần rà soát phần test\n"
            "- Kế hoạch tuần tới: hoàn thiện test + release note\n"
        )
    else:
        draft = (
            f"BÁO CÁO TUẦN - BẢN CHỈNH SỬA LẦN {revision_count + 1}\n"
            f"- Yêu cầu gốc: {base_request}\n"
            f"- Feedback reviewer: {feedback}\n"
            "- Tiến độ: Hoàn thành 90%\n"
            "- Rủi ro: Đã làm rõ phần test và release\n"
            "- Kế hoạch tuần tới: finalize test + phát hành\n"
        )

    return {
        "draft": draft,
        "approved": False,
        "last_decision": "generated"
    }


def human_review_node(state: State):
    """
    Dừng graph để human review.
    Nếu resume bằng approve/edit/reject thì giá trị đó là return value của interrupt(...)
    """
    decision = interrupt(
        {
            "kind": "review_report",
            "draft": state["draft"],
            "revision_count": state.get("revision_count", 0),
            "allowed_decisions": ["approve", "edit", "reject"],
            "message": "Hãy review report. Nếu edit, nhập feedback để hệ thống generate lại."
        }
    )

    # decision ví dụ:
    # {"type": "approve", "note": "OK"}
    # {"type": "edit", "feedback": "Bổ sung phần rủi ro và viết ngắn hơn", "note": "Cần sửa"}
    # {"type": "reject", "note": "Không dùng báo cáo này"}

    decision_type = decision["type"]

    if decision_type == "approve":
        return {
            "approved": True,
            "reviewer_note": decision.get("note", "Approved by human"),
            "last_decision": "approve"
        }

    if decision_type == "edit":
        return {
            "approved": False,
            "feedback": decision["feedback"],
            "reviewer_note": decision.get("note", "Edited by human"),
            "last_decision": "edit"
        }

    return {
        "approved": False,
        "reviewer_note": decision.get("note", "Rejected by human"),
        "last_decision": "reject"
    }


def apply_feedback_node(state: State):
    """
    Áp feedback vào state và tăng revision_count.
    Sau đó graph quay lại generate_report_node để generate lại report.
    """
    return {
        "revision_count": state.get("revision_count", 0) + 1,
        "last_decision": "feedback_applied"
    }


def route_after_review(state: State):
    """
    Routing sau human review:
    - approve -> save_report_node
    - edit -> apply_feedback_node -> generate_report_node
    - reject -> END
    """
    if state["last_decision"] == "approve":
        return "save_report_node"

    if state["last_decision"] == "edit":
        return "apply_feedback_node"

    return END


def save_report_node(state: State):
    """
    Chỉ được chạy khi human approve.
    """
    path = "approved_weekly_report.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write(state["draft"])

    return {
        "saved_path": path,
        "last_decision": "saved"
    }


builder = StateGraph(State)

builder.add_node("generate_report_node", generate_report_node)
builder.add_node("human_review_node", human_review_node)
builder.add_node("apply_feedback_node", apply_feedback_node)
builder.add_node("save_report_node", save_report_node)

builder.add_edge(START, "generate_report_node")
builder.add_edge("generate_report_node", "human_review_node")

builder.add_conditional_edges("human_review_node", route_after_review)

builder.add_edge("apply_feedback_node", "generate_report_node")
builder.add_edge("save_report_node", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)
``

In [ ]:
config = {
    "configurable": {
        "thread_id": "hitl-demo-001"
    }
}

# Lần chạy đầu: graph sẽ dừng ở human_review_node
result = graph.invoke(
    {"request": "Viết báo cáo tuần thật ngắn gọn, rõ ràng"},
    config=config
)

print(result)
# Tùy runtime / cách gọi, bạn sẽ thấy graph đã dừng tại interrupt.
# Khi deploy qua Agent Server / LangGraph Server, interrupt payload sẽ xuất hiện rõ ràng
# trong kết quả run như docs minh họa.

In [ ]:
from langgraph.types import Command

# Case 1: approve nguyên trạng
resume_result = graph.invoke(
    Command(
        resume={
            "type": "approve",
            "note": "OK, lưu file đi"
        }
    ),
    config=config
)

print(resume_result)
# Kết quả mong đợi:
# - approved = True
# - saved_path = "approved_weekly_report.txt"

In [ ]:
# Case 2: edit rồi mới approve
resume_result = graph.invoke(
    Command(
        resume={
            "type": "edit",
            "draft": (
                "BÁO CÁO TUẦN\n"
                "- Tiến độ: Hoàn thành 90%\n"
                "- Rủi ro: Không đáng kể\n"
                "- Kế hoạch: Đóng release trong tuần này\n"
            ),
            "note": "Đã chỉnh lại wording cho ngắn gọn hơn"
        }
    ),
    config=config
)

print(resume_result)

In [ ]:
# Case 3: reject
resume_result = graph.invoke(
    Command(
        resume={
            "type": "reject",
            "note": "Bổ sung thêm thông tin test rồi quay lại"
        }
    ),
    config=config
)

print(resume_result)
# Kết quả mong đợi:
# - graph kết thúc mà không save file